# 03a — End-to-end pipeline comparison (replay)

This notebook is the **industry demo**: 6 different vertex-coloring
solvers — greedy heuristic, exact ILP (HiGHS + optional Hexaly),
classical column generation, **quantum** column generation (Dirac
replayed), and classical Branch-and-Price (single-child IS-fixing) — on
the bundled ER(20, 0.7) instance.

The math is hidden; what we compare is **χ**, runtime, columns/call,
and final coloring quality.

For a live version with your own graph, see `03b`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import _demo_utils as U
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time

from quantum_colgen.column_generation import column_generation
from quantum_colgen.pricing.classical_lp import ClassicalLPPricingOracle
from quantum_colgen.direct_ilp import solve_coloring_ilp_highs


## 1. The instance

In [ ]:
psp = U.load_psp(1)
G = U.psp_to_graph(psp)
layout = U.psp_to_layout(psp)

print(f"Graph: {psp['instance_id']}   n={psp['n']}   m={psp['m']}")
fig, ax = plt.subplots(figsize=(5.8, 4.0), constrained_layout=True)
U.draw_graph(ax, G, layout, node_color="white",
             title=f"{psp['instance_id']}  ({psp['n']} vertices, {psp['m']} edges)")
plt.show()


## 2. Run all six solvers

In [ ]:
results = {}

# (a) Greedy heuristic — DSATUR via networkx
t0 = time.monotonic()
greedy_color = nx.coloring.greedy_color(G, strategy="DSATUR")
wall = time.monotonic() - t0
classes = {}
for v, c in greedy_color.items():
    classes.setdefault(c, set()).add(v)
greedy_coloring = list(classes.values())
results["Greedy (DSATUR)"] = {
    "chi": len(greedy_coloring), "runtime_s": wall,
    "iterations": None, "columns_per_call": None, "api_calls": None,
    "coloring": greedy_coloring,
}
print(f"Greedy (DSATUR): χ={len(greedy_coloring)}  wall={wall*1000:.1f}ms")


In [ ]:
# (b) HiGHS exact ILP
t0 = time.monotonic()
chi_h, coloring_h, _, info_h = solve_coloring_ilp_highs(G, time_limit=60)
wall = time.monotonic() - t0
results["HiGHS exact ILP"] = {
    "chi": chi_h, "runtime_s": wall, "iterations": None,
    "columns_per_call": None, "api_calls": None,
    "coloring": coloring_h or [],
    "notes": "optimal" if info_h.get("optimal") else "best-found",
}
print(f"HiGHS exact ILP: χ={chi_h}  wall={wall:.2f}s  "
      f"{'(optimal)' if info_h.get('optimal') else '(best-found)'}")


In [ ]:
# (c) Hexaly exact ILP — graceful skip if not importable
try:
    from quantum_colgen.direct_ilp import solve_coloring_ilp_hexaly
    t0 = time.monotonic()
    chi_x, coloring_x, _, info_x = solve_coloring_ilp_hexaly(G, time_limit=60)
    wall = time.monotonic() - t0
    results["Hexaly exact ILP"] = {
        "chi": chi_x, "runtime_s": wall, "iterations": None,
        "columns_per_call": None, "api_calls": None,
        "coloring": coloring_x or [],
        "notes": "optimal" if info_x.get("optimal") else "best-found",
    }
    print(f"Hexaly exact ILP: χ={chi_x}  wall={wall:.2f}s")
except (ImportError, RuntimeError, Exception) as e:
    print(f"Hexaly skipped: {type(e).__name__}: {e}")
    print("(Set DYLD_LIBRARY_PATH and PYTHONPATH per CLAUDE.md to enable Hexaly.)")


In [ ]:
# (d) Classical CG (LP-based pricing oracle)
classical_oracle = ClassicalLPPricingOracle()
t0 = time.monotonic()
chi_c, coloring_c, stats_c = column_generation(
    G, classical_oracle, max_iterations=200, verbose=False)
wall = time.monotonic() - t0
results["Classical CG"] = {
    "chi": chi_c, "runtime_s": wall,
    "iterations": stats_c.get("iterations"),
    "columns_per_call": classical_oracle.timer.summary().get('avg_columns_per_call'),
    "api_calls": classical_oracle.timer.summary().get('num_api_calls'),
    "coloring": [set(c) for c in coloring_c],
}
print(f"Classical CG: χ={chi_c}  wall={wall:.2f}s  "
      f"iters={stats_c.get('iterations')}  "
      f"cpc={results['Classical CG']['columns_per_call']:.1f}")


In [ ]:
# (e) Quantum CG — Dirac replayed from bundled raw samples
qc_oracle = U.make_dirac_oracle("replay", method="gibbons",
                                multi_prune=True, randomized_rounding=True)
t0 = time.monotonic()
chi_q, coloring_q, stats_q = column_generation(
    G, qc_oracle, max_iterations=200, verbose=False)
wall = time.monotonic() - t0
results["Quantum CG (Dirac, replay)"] = {
    "chi": chi_q, "runtime_s": wall,
    "iterations": stats_q.get("iterations"),
    "columns_per_call": qc_oracle.timer.summary().get('avg_columns_per_call'),
    "api_calls": qc_oracle.timer.summary().get('num_api_calls'),
    "coloring": [set(c) for c in coloring_q],
    "notes": "Dirac responses replayed from RF-branching/instances/...",
}
print(f"Quantum CG: χ={chi_q}  wall={wall:.2f}s (replay)  "
      f"iters={stats_q.get('iterations')}  "
      f"cpc={results['Quantum CG (Dirac, replay)']['columns_per_call']:.1f}")


In [ ]:
# (f) Classical Branch-and-Price (single-child IS-fixing branching)
import subprocess, json, tempfile

QBP_DIR = U.REPO_ROOT / "quantum-branch-price"
qbp_run = QBP_DIR / "scripts" / "run_bp.py"

def try_classical_bp(graph, time_limit=60):
    if not qbp_run.exists():
        raise RuntimeError(f"qbp run_bp.py not found at {qbp_run}")
    with tempfile.TemporaryDirectory() as td:
        out = Path(td) / "result.json"
        # The qbp CLI accepts named graphs or er_N_P_sS; we use the latter
        # to match the bundled instance.
        proc = subprocess.run(
            ["uv", "run", "python", str(qbp_run),
             "--graph", "er_20_0.7_s0",
             "--branching", "is_fixing",
             "--oracle", "classical-lp",
             "--time-limit", str(time_limit),
             "--ilp-time-limit", "30",
             "--json", str(out)],
            cwd=str(QBP_DIR), capture_output=True, text=True,
            timeout=time_limit + 60,
        )
        if proc.returncode != 0:
            raise RuntimeError(f"qbp returned {proc.returncode}: "
                               f"{proc.stderr[-500:]}")
        if not out.exists():
            raise RuntimeError("qbp wrote no JSON")
        return json.loads(out.read_text()), proc.stdout

try:
    t0 = time.monotonic()
    rec, stdout = try_classical_bp(G, time_limit=60)
    if isinstance(rec, list) and rec:
        rec = rec[0]
    if not isinstance(rec, dict):
        raise RuntimeError(f'unexpected qbp schema: {type(rec).__name__}')
    wall = time.monotonic() - t0
    chi_bp = rec.get("chi") or next((m.get("chi") for m in rec.get("methods", {}).values()), None)
    nodes_explored = next((m.get("nodes_explored") for m in rec.get("methods", {}).values()), None)
    results["Classical B&P (IS-fixing)"] = {
        "chi": chi_bp, "runtime_s": wall, "iterations": nodes_explored,
        "columns_per_call": None, "api_calls": None,
        "coloring": [],  # extracted by qbp; not loaded here
        "notes": f"{nodes_explored} B&B nodes",
    }
    print(f"Classical B&P: χ={chi_bp}  wall={wall:.2f}s  "
          f"nodes={nodes_explored}")
except Exception as e:
    print(f"Classical B&P skipped: {type(e).__name__}: {e}")
    print("(Set up the quantum-branch-price submodule + venv to enable.)")


## 3. Comparison table

In [ ]:
# Build a clean DataFrame
df = pd.DataFrame([
    {"solver": k, "χ": v["chi"], "runtime_s": round(v["runtime_s"], 2),
     "iterations": v.get("iterations"),
     "cols/call": (round(v["columns_per_call"], 2)
                   if v.get("columns_per_call") else None),
     "api_calls": v.get("api_calls"),
     "notes": v.get("notes", "")}
    for k, v in results.items()
])
df


## 4. Coloring side-by-side

In [ ]:
solvers_with_coloring = [(k, v) for k, v in results.items()
                          if v.get("coloring")]
ncols = 3
nrows = (len(solvers_with_coloring) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows),
                         constrained_layout=True)
axes = np.atleast_2d(axes).reshape(nrows, ncols)
for k, (name, v) in enumerate(solvers_with_coloring):
    r, c = divmod(k, ncols)
    U.draw_coloring(G, v["coloring"], layout, ax=axes[r, c],
                    title=f"{name}\nχ={v['chi']}")
for k in range(len(solvers_with_coloring), nrows * ncols):
    r, c = divmod(k, ncols)
    axes[r, c].set_visible(False)
plt.show()


In [ ]:
# Runtime bar chart
fig, ax = plt.subplots(figsize=(7.5, 3.0), constrained_layout=True)
labels = list(results.keys())
times = [results[k]["runtime_s"] for k in labels]
colors = [U.QCI_GREEN if "Greedy" in k else
          (U.QCI_BLUE if "Classical" in k or "HiGHS" in k or "Hexaly" in k else U.QCI_ORANGE)
          for k in labels]
ax.barh(labels, times, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xscale("log")
ax.set_xlabel("runtime (seconds, log scale)")
ax.set_title(f"Runtime comparison on {psp['instance_id']}")
plt.show()


## 5. Takeaway

* **HiGHS / Hexaly** typically prove optimum (or get very close) on graphs
  this small in seconds.
* **Classical CG** matches optimum and surfaces a *dual bound* via the
  RMP; it scales to larger graphs where direct ILP runs out of budget.
* **Quantum CG (replay)** matches the classical CG χ here, with a
  different column-generation pattern (fewer, larger columns per call).
  On the recorded device these calls were ~30–90 seconds each — the
  replay completes in milliseconds because the device work was done
  upstream.
* **Classical B&P** closes any remaining integrality gap when CG alone
  doesn't reach the optimum — useful on harder/denser graphs.

For a notebook where you can swap in your own graph and watch Dirac
work in real time, see `03b`.